# Let subset of cells learn

A code for doing that.

In [ ]:
p = 7  # Probability of cells undergoing training (The non-hidden units)


########## If we want to check for many different fractions 
fractions_cL = []

cL_increment = 0
while not (cL_increment > 100):
    fractions_cL.append(cL_increment)
    cL_increment += 5

fractions_cL = [x/100 for x in fractions_cL]  # will be passed as argument in the if compute_for_all == True part later




[0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]


In [ ]:
import numpy as np 
p = 0.7
N = 200
nLearn = np.floor(p*N).astype(int)
allCells = [i for i in range(1, (N + 1))]

# permutated = np.random.permutation(allCells)


# cL = permutated[0:allCells]
# ncL = permutated[allCells:N]
cL = allCells[0:nLearn]
ncL = allCells[nLearn:N]

print(allCells)
print(cL)
print(ncL)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200]
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29,

In [ ]:
import numpy as np 
p = 0.7
N = 200
nLearn = np.floor(p*N).astype(int)  # number of neurons who will undergo training
allCells = [i for i in range(1, (N + 1))]

permutated = np.random.permutation(allCells)
cL = permutated[0:allCells]
ncL = permutated[allCells:N]
# cL = allCells[0:nLearn]
# ncL = allCells[nLearn:N]

print(allCells)
print(cL)
print(ncL)

nLearn = cL.shape[1]  # How many cells undergo learning
#no_of_nonzero_con = sum(isinstance(lst, list) for lst in nonzero_con)


def initialize_connectivity_matrix(N, p, gsyn):
    w = np.random.randn(N, N)  # fully connected
    # w = sparse.random(N, N, p, data_rvs=np.random.randn).todense()
    np.fill_diagonal(w, 0)  # No autapses
    w *= gsyn / np.sqrt(p * N)
    
    for i in range(N):
        i0 = np.where(w[i, :])[1]
        if len(i0) > 0:
            av0 = np.sum(w[i, i0]) / len(i0)
            w[i, i0] -= av0
    
    return w

def initialize_neurons(N):
    x = np.random.uniform(size=N) * 2 * np.pi
    r = np.zeros(N)
    nspike = np.zeros(N)
    return x, r, nspike

def initialize_training(N, w):
    # Initialize correlation matrices for RLS learning
    nind=np.zeros(N).astype('int')
    idx=[]
    P=[]
    for i in range(N):
        ind=np.where(w[i,:])[1]
        nind[i]=len(ind)
        idx.append(ind)
        P.append(np.identity(nind[i])/alpha)   
    return P, idx

# def currents(N, itmax):
#     Iext=np.zeros((N,itmax))
#     Ibac=amp_corriente*(2*np.random.uniform(size=N)-1)
#     Iext[:, :itstim] = Ibac[:, None]  # Vectorized assignment
#     return Iext


def learning(it, iloop, w, r, P, idx, target, norm_w0, csv_writer):
    error = target[:, it:it + 1] - w @ r.reshape(N, 1)
    # for i in range(N):
    for i in range(cL):
        ri = r[idx[i]].reshape(len(idx[i]), 1)  
        k1 = P[i] @ ri
        k2 = ri.T @ P[i]
        den = 1 + ri.T @ k1
        P[i] -= (k1 @ k2) / den
        dw = error[i, 0] * P[i] @ r[idx[i]]
        w[i, idx[i]] += dw

    if it % 10 == 0:
        modt_value = it + iloop * itmax
        modw_value = np.log(np.linalg.norm(w) / norm_w0)
        csv_writer.writerow([modt_value, modw_value])
        
    return w, P